# 01 — OCR & Semantic Extraction Pipeline
Main production pipeline: PDF pages in, semantic CSV + OCR text out.


## Imports
Image/PDF libraries, API key, Drive folders.


In [ ]:
# Imports
from PIL import Image
from pathlib import Path
import base64
import io
import subprocess
import shutil
from io import BytesIO
from langchain_openai import ChatOpenAI
import csv
from io import StringIO
import re
from pdf2image import convert_from_path
import time
import sys
import logging
from googleapiclient.http import MediaIoBaseUpload


# LOAD API KEY

from config import (
    OPENAI_KEY_FILE,
    PAGES_DIR,
    OCR_RESULTS_DIR,
    LOGS_DIR,
    PDF_PATH,
    PROGRESS_FILE,
    DRIVE_ROOT_FOLDER,
    SAVE_MODE,
)

api_file = Path(OPENAI_KEY_FILE)
with open(api_file, "r") as f:
    cont = f.readlines()
params = dict(v.strip().split("=", 1) for v in cont if "=" in v)
OPENAI_API_KEY = params["api_key"]
llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)


# FOR DRIVE PURPOSES

from drive_utils import (
    get_drive_service,
    get_or_create_folder,
    get_or_create_folder as get_or_create_drive_folder,
    upload_file as upload_to_drive,
    upload_with_retry,
    drive_file_exists,
    list_files_in_folder,
    download_bytes,
    save_or_upload_text,
)

def setup_drive_folders(service, root_folder_id: str) -> dict:
    # creates images/ and llm_ocr_results/ under root, returns their ids
    return {
        "images": get_or_create_folder(service, "images", root_folder_id),
        "ocr": get_or_create_folder(service, "llm_ocr_results", root_folder_id),
    }


## Configuration
Run-mode toggle: full corpus, specific pages, or a page range.


In [2]:
# RUN CONFIGURATION

# Mode options:
# "full"  — process entire PDF (default)
# "pages"  — process specific pages listed in SELECTED_PAGES
# "range"  — process a range of page numbers

RUN_MODE = "full"            # "full" | "pages" | "range"
SELECTED_PAGES = [458]    # used when RUN_MODE = "pages"
PAGE_RANGE = (494, 570)            # used when RUN_MODE = "range"
FORCE_REPROCESS = False          # if True, re-runs even if page is in progress.txt


## Helper functions
Image encoding, logging, PDF extraction, progress tracking, CSV utilities, run numbering.


In [ ]:
#image encoding

def encode_image(img):
    if img.mode != "RGB":
        img = img.convert("RGB")
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode()


In [ ]:
# Creates a run-numbered log file, checking local files and Drive.
def setup_logging(log_dir: Path = None, run_mode: str = "full",
                  selected_pages: list = None, page_range: tuple = None,
                  drive_service=None, logs_folder_id: str = None) -> Path:
    # sets up stdout + run-numbered log file, run_N_scope.log
    log_dir = log_dir or (Path.cwd() / "logs")
    log_dir.mkdir(exist_ok=True)

    if run_mode == "pages" and selected_pages:
        scope = "pages_" + "_".join(map(str, sorted(selected_pages)))
    elif run_mode == "range" and page_range:
        scope = f"range_{page_range[0]}_{page_range[1]}"
    else:
        scope = "all"

    # Check local logs
    max_n = 0
    existing = list(log_dir.glob("run_*.log"))
    for f in existing:
        m = re.match(r'^run_(\d+)_', f.name)
        if m:
            max_n = max(max_n, int(m.group(1)))

    # Also check Drive logs folder if available
    if drive_service and logs_folder_id:
        try:
            for f in list_files_in_folder(drive_service, logs_folder_id):
                if 'run_' not in f["name"]:
                    continue
                m = re.match(r'^run_(\d+)_', f["name"])
                if m:
                    max_n = max(max_n, int(m.group(1)))
        except Exception:
            pass  # Drive check failed, use local count only

    n = max_n + 1
    log_path = log_dir / f"run_{n}_{scope}.log"

    class TeeHandler(logging.StreamHandler):
        def __init__(self, file_path):
            super().__init__(sys.stdout)
            self.file_handler = logging.FileHandler(file_path, encoding="utf-8")
            self.file_handler.setFormatter(logging.Formatter("%(message)s"))

        def emit(self, record):
            super().emit(record)
            self.file_handler.emit(record)

    logger = logging.getLogger("ocr_pipeline")
    logger.setLevel(logging.INFO)
    for h in list(logger.handlers):
        if hasattr(h, "file_handler"):
            h.file_handler.close()
        logger.removeHandler(h)
    logger.addHandler(TeeHandler(log_path))

    return log_path


#limit the time between 2 llm calls
_last_call_time = 0.0
MIN_SECONDS_BETWEEN_CALLS = 0.5  # adjust based on API tier

def rate_limited_invoke(llm, messages):
    # wraps llm.invoke with rate limiting
    global _last_call_time
    elapsed = time.time() - _last_call_time
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)
    result = llm.invoke(messages)
    _last_call_time = time.time()
    return result



In [ ]:
# Converts PDF pages to JPEG and uploads them to Drive, skipping pages already there.
def extract_pdf_pages(pdf_path: Path, dpi: int = 200,
                      drive_service=None,
                      drive_folder_id=None,
                      page_filter: set = None) -> list[tuple[str, str]]:
    # PDF -> JPEG pages, uploaded straight to Drive, no local storage
 
    logger = logging.getLogger("ocr_pipeline")
 
    # We need the total page count first without converting everything.
    # convert_from_path with first_page/last_page avoids loading unneeded pages.
 
    def _get_page_count(pdf_path):
        # fast page count via pdfinfo, with fallback
        if shutil.which("pdfinfo"):
            try:
                out = subprocess.check_output(["pdfinfo", str(pdf_path)], text=True)
                for line in out.splitlines():
                    if line.startswith("Pages:"):
                        return int(line.split(":")[1].strip())
            except Exception:
                pass
        # Fallback: convert a dummy page to get count from convert_from_path metadata
        pages = convert_from_path(pdf_path, dpi=10, fmt="jpeg")
        return len(pages)
 
    total = _get_page_count(pdf_path)
    logger.info(f"PDF has {total} pages. Filter: {sorted(page_filter) if page_filter else 'all'}")
 
    results = []
 
    for i in range(1, total + 1):
        page_name = f"page_{i}.jpg"
 
        # Skip pages outside the filter entirely 
        if page_filter and i not in page_filter:
            results.append((page_name, None))
            continue
 
        # Check Drive for existing file
        if drive_service and drive_folder_id:
            existing_id = drive_file_exists(drive_service, drive_folder_id, page_name)
            if existing_id:
                logger.info(f"Skipping existing: {page_name}")
                results.append((page_name, existing_id))
                continue
 
        # Convert only this single page
        logger.info(f"Converting {page_name} at {dpi} DPI...")
        pages = convert_from_path(pdf_path, dpi=dpi, fmt="jpeg",
                                  first_page=i, last_page=i)
        if not pages:
            logger.info(f"  -> Warning: no image returned for page {i}, skipping.")
            results.append((page_name, None))
            continue
 
        page = pages[0]
        img_buffer = io.BytesIO()
        page.save(img_buffer, format="JPEG", quality=95)
        img_buffer.seek(0)
 
        drive_id = None
        if drive_service and drive_folder_id:
            metadata = {"name": page_name, "parents": [drive_folder_id]}
            media = MediaIoBaseUpload(img_buffer, mimetype="image/jpeg",
                                      resumable=True, chunksize=1024 * 1024)
            file = upload_with_retry(drive_service, body=metadata, media=media)
            drive_id = file["id"]
            time.sleep(0.3)
 
        results.append((page_name, drive_id))
        del page
        img_buffer.close()
 
    logger.info(f"Page list ready: {len([r for r in results if r[1]])} on Drive, "
                f"{len([r for r in results if r[1] is None])} skipped/missing.")
    return results


In [ ]:
#load and mark progress for batch processing
def load_progress(progress_file: Path) -> set:
    # page names already successfully processed
    if not progress_file.exists():
        return set()
    try:
        return set(progress_file.read_text(encoding="utf-8").splitlines())
    except:
        return set()

def mark_progress(progress_file: Path, page_name: str):
    # marks a page as successfully processed
    with open(progress_file, "a", encoding="utf-8") as f:
        f.write(page_name + "\n")

PROVINCIA_MODE = True     # extract provincia intro data for pages 494-570


In [ ]:
def clean(text: str):
    # strips markdown artifacts like ```csv and whitespace
    return text.replace("```csv", "").replace("```", "").strip()


#save version drive
def save(path: Path, text: str, drive_service=None, drive_folder_id: str = None):
    # saves text locally and/or to Drive per SAVE_MODE
    save_or_upload_text(text, path, drive_service, drive_folder_id)
# Pulls the numeric page number out of a page/file name.
def extract_page_number(page_name):
    match = re.search(r"\d+", page_name)
    return int(match.group()) if match else ""

# Appends a page-number column to a semantic CSV.
def add_page_column(csv_text: str, page_num) -> str:
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if not lines:
        return csv_text

    output = StringIO()
    writer = csv.writer(output)
    reader = csv.reader(lines)
    rows = list(reader)

    if not rows:
        return csv_text

    header = rows[0]
    n_cols = len(header)

    if "page" not in [h.strip().lower() for h in header]:
        writer.writerow(header + ["page"])
        for row in rows[1:]:
            if len(row) < n_cols:
                row = row + [""] * (n_cols - len(row))
            writer.writerow(row + [page_num])
    else:
        writer.writerow(header)
        for row in rows[1:]:
            writer.writerow(row)

    return output.getvalue().strip()


def repair_csv(csv_text: str, expected_columns: int = None) -> str:
    # re-parses/rewrites CSV to normalize quoting, merges overflow columns back
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if not lines:
        return csv_text

    output = StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    reader = csv.reader(lines)
    rows = list(reader)

    if not rows:
        return csv_text

    header = rows[0]
    n_cols = expected_columns or len(header)
    writer.writerow(header)

    for row in rows[1:]:
        if len(row) > n_cols:
            fixed = row[:n_cols - 1] + [", ".join(row[n_cols - 1:])]
            writer.writerow(fixed)
        else:
            writer.writerow(row)

    return output.getvalue().strip()


In [ ]:
#get_next_run_id depends on the files in the drive
def get_next_run_id(page_dir: Path, page_name: str,
                    drive_service=None, drive_folder_id: str = None) -> int:
    # next run id, checking local and/or Drive per SAVE_MODE
    max_id = 0

    if SAVE_MODE in ("local", "both"):
        existing = list(page_dir.glob(f"{page_name}_ocr_*.txt"))
        for f in existing:
            try:
                max_id = max(max_id, int(f.stem.split("_")[-1]))
            except ValueError:
                pass

    if SAVE_MODE in ("drive", "both") and drive_service and drive_folder_id:
        for f in list_files_in_folder(drive_service, drive_folder_id):
            if f"{page_name}_ocr_" not in f["name"]:
                continue
            try:
                max_id = max(max_id, int(f["name"].replace(".txt","").split("_")[-1]))
            except ValueError:
                pass

    return max_id + 1


# Next version number for a provincia aggregate file.
def get_next_provincia_version(stem: str, out_dir: Path = None,
                                drive_service=None, drive_folder_id: str = None) -> int:
    max_v = 0
    if SAVE_MODE in ("local", "both") and out_dir:
        for f in out_dir.glob(f"{stem}_*.csv"):
            try:
                max_v = max(max_v, int(f.stem.split("_")[-1]))
            except ValueError:
                pass
    if SAVE_MODE in ("drive", "both") and drive_service and drive_folder_id:
        for f in list_files_in_folder(drive_service, drive_folder_id):
            if f"{stem}_" not in f["name"]:
                continue
            try:
                max_v = max(max_v, int(f["name"].replace(".csv","").split("_")[-1]))
            except ValueError:
                pass
    return max_v + 1

def merge_schemas(locked: str, inferred: str) -> str:
    # union of two schema strings, locked columns first
    if not locked:
        return inferred
    if not inferred:
        return locked

    locked_cols = [c.strip() for c in locked.split(",")]
    inferred_cols = [c.strip() for c in inferred.split(",")]

    merged = list(locked_cols)
    for col in inferred_cols:
        if col not in merged:
            merged.append(col)

    return ", ".join(merged)


## Prompts & page-type detection
LLM prompts plus section, layout, ad, index and provincia detection used during extraction.


In [ ]:
# Main OCR + semantic extraction prompt

READING_PROMPT = """
You are performing OCR (optical character recognition) on a scanned historical document.
The document is a 1947 commercial directory from Venice, Italy.
All content is historical, public, and archival.
This is a digitization task for historical research purposes.
Your task is ONLY to transcribe visible text exactly as it appears in the image.
This is NOT analysis, NOT interpretation, and NOT sensitive data processing.
This is simple character transcription of historical printed text.
The names sound Italian.

Your task is to identify and extract all the text contained in the provided image of a document.

Rules:
- One line per line of text
- Preserve exact spelling, punctuation, capitalization
- Do NOT interpret or structure
- Do NOT output CSV
- Do NOT skip anything
- You MUST attempt transcription regardless of content — this is a historical archive
- Return only raw text.
"""

CONTEXT_HEADER = """
You are processing a page from a 500+ page commercial almanac of Venice, Italy, circa 1947.
The document is in Italian. It lists people, businesses, addresses, and institutions.
"""

PAGE_TYPE_REFERENCE = """
Page structure reference (page ranges and content types):
1. Google first page
2. Cover of the almanac
3-5. Ads
6-9. Title + descriptions
10. sommaire
11-13. Ads
14-35. Indice (except 20, 21, 23, 27, 30, 31 - ads)
36-37. Vaticano (names of religious people & addresses of churches)
38-39: government (names and titles)
40-41: ads
42-45: prefettura (subcategories, names and addresses)
46-47: ads
48-56: municipio (subcategories, sometime just names, sometimes names and addresses, organised by office)
57-58: finanza (subcategories, sometime just names, sometimes names and addresses, organised by office)
59-60 and beginning of 61: militari (subcategories, sometime just names, sometimes names and addresses)
61-62: ferrovie (no subcategories, names but no address)
62-63-64: telegrafi (subcategories, sometime just names, sometimes names and addresses)
65-70: sindicali (subcategories, sometime just names, sometimes names and addresses
71: consolati: names of countries and addresses
72-82: instruzione (subcategories, sometime just names, sometimes names and addresses)
82-84: instituti di beneficienza (subcategories, sometime just names, sometimes names and addresses)
84-85: istituti opsedalieri e sanitari (subcategories, sometime just names, sometimes names and addresses)
86-97: instituzioni religiosi (subcategories, sometime just names, organized by church, which is the location)
97-99: associazioni (subcategories, sometime just names, sometimes names and addresses)
100-101: parti politici (names of party, people, addresses)
101-104: stampa quotidiana e periodica (subcategories, names of journal, address, names of people working there)
105-107: ads
108 - 128 : proffessionisti e artisti: list of names and address, by subcategory of work, classified by alphabetical order, sometimes ads
129-131: ads
132- 318: industria e commercio : list of names and address, by subcategory of work or object of commerce, classified by alphabetical order, sometimes also subcategorised by neigborhood, sometimes ads
319 - 331: ads
332 - 487:  indice general alfabetico : list of names and addresses, organized by alfabetical order (of all the document), could be names of names of things (restaurants, places...)
490: title "provincia di venezia"
491-493: index (elenco)
494-570: provincia di venezia, organized by places and lists common places, sometimes with names and addresses
572-575: ads
"""

# Document section boundaries — page numbers where a new major section starts.
# Cross-page section_state resets at these boundaries.
DOCUMENT_SECTION_BOUNDARIES = {
    1, 2, 3, 6, 10, 11, 14, 36, 38, 40, 42, 46, 48, 57, 59, 61, 62, 65,
    71, 72, 82, 84, 86, 97, 100, 101, 105, 108, 129, 132, 319, 332,
    490, 491, 494, 572
}

SEMANTIC_PROMPT_FROM_TEXT = """
Return a CSV where:
- One entity per row
- The FIRST row is the header
- Any field containing a comma MUST be wrapped in double quotes. Example: "Rossi, Bianchi, Verdi"
- This applies especially to committee/group names like "Comitato di Redazione: X, Y, Z", and to addresses.
- Never split a single field across multiple columns due to internal commas
- CRITICAL NAME RULE: The Name field must contain the person's COMPLETE name including:
  title if it exists (Dott., Rag., Ing., Prof., Avv., Cav., etc.), first name, and last name — all in one field.
  Never truncate a name at a comma or period. Never put only a title like "Dott" in Name.
  If the document lists surname first then first name (e.g. "Cicogna ing. Giovanni"), combine them as written into one Name field: "Cicogna ing. Giovanni".
  If the name appears split across the line (title on one part, name on another), reconstruct the full name in the Name column.
- An address often contains a street and a street number.
- An address can also be a reference to a church, a basilica, a palazzo, a convento, a specific building.
- Every entry should have either an address or a location (or both) (even if just the name of a church or building or general location)
- CRITICAL ADDRESS RULE: only fill the Address column if an address is EXPLICITLY stated for that entry in the text.
  If no address is stated for an entry, leave Address EMPTY. Do not copy an address from a nearby entry.
  Address propagation is handled separately — your job is only to transcribe what is explicitly written.
- Ensure Address and Location can coexist when both are present in the text. Do not force a single-column interpretation.
- Ensure Role and Profession can coexist when both are present in the text. Do not force a single-column interpretation.
- Leave fields empty if missing
- Transcribe exactly: preserve spelling, punctuation
- Return ONLY CSV, no extra text

Text:
"""


SEMANTIC_PROMPT_FROM_IMAGE_AND_TEXT = """
You are extracting structured data from a page of a 1947 Venetian almanac.
You have been provided BOTH the scanned image AND the OCR transcription of this page.

Use the OCR text as your primary source for exact spelling of names and addresses.
Use the image to understand the VISUAL STRUCTURE: which entries belong to which section,
where section headings appear, and which address or category applies to which block of entries.

The page has complex layout with multiple sections and possibly advertisements.
Advertisements appear in visually distinct bordered boxes — extract them as normal rows.

Return a CSV using the schema provided.
Follow all section inference rules:
- Section headings get a __SECTION__ row with their embedded data
- Entries follow their section's __SECTION__ row
- Do NOT carry forward values yourself

Return ONLY CSV, no extra text.
"""


In [ ]:
# Refusal detection
REFUSAL_PHRASES = [
    "unable to assist",
    "unable to provide",
    "unable to transcribe",
    "can't help",
    "cannot help",
    "i'm sorry",
    "i cannot",
    "not able to",
    "can't assist with that",
    "i am unable",
    "i'm unable",
    "cannot provide",
    "cannot transcribe",
    "cannot process",
    "i cannot provide",
    "i cannot transcribe"
]

# Checks text against known LLM-refusal phrases.
def is_refusal(text):
    return any(p.lower() in text.lower() for p in REFUSAL_PHRASES)

FALLBACK_PROMPT = """
Extract all entries from this text as CSV.
Columns: Name, Address, Category, Notes
- Addresses may contain commas. Always keep the full address in a single field. If needed, use quotes around the full Address field. If needed, split the address field between Address and Location.
One row per entry. Return ONLY CSV, no extra text.

Text:
"""


In [ ]:
# Asks the LLM to propose CSV column names for a page
SCHEMA_PROMPT = """
You are looking at raw text from a page of a 1947 Venetian commercial almanac in Italian.
What CSV columns would best capture the structured data on this page?
Return only a comma-separated list of column names, nothing else.
The first column MUST always be Name (the person's full name, never a title or role).
Always include an Address column.
Always include a Location column for sestiere, neighborhood, sub-location, or physical section references.
If professions or occupations are present, include a Profession column (broader occupation or descriptive category)
If roles or job titles are present, include a Role column. They are functions or positions (e.g. Direttore, Capo Servizio, Ispettore, Segretario). Do not merge Role into Profession.
If it is a list of journals or political party, add a column for it.
Default if unsure: Name,Address,Location,Role,Profession,Additional Info
Return only the comma-separated list of columns.

Text:
"""

# Asks the LLM which columns this page needs.
def infer_schema(raw_text):
    msg = [{"role": "user", "content": SCHEMA_PROMPT + "\n" + raw_text}]
    res = rate_limited_invoke(llm,msg)
    return clean(str(res.content))


In [ ]:
# Detects multiple distinct sections on a page
SECTION_DETECT_PROMPT = """
Look at this text from a 1947 Venetian almanac page.
Does it contain multiple distinct sections with different structures?
If yes, list each section, one per line, in this format:
  SECTION_TITLE | field: value, field: value
Where field:value pairs capture any address, category, or other data embedded in the section title line.
Example: PREFETTURA | Address: Palazzo del Governo - S. Maurizio fondamenta Zaguri 2662
Example: PERIODICI DI CARATTERE RELIGIOSO | Character: Periodici di carattere religioso
If no embedded data, just list the title alone:
  Gabinetto
If no sections at all, return exactly: SINGLE_SECTION

Rules:
LEVEL 1: ORGANIZATION (top-level, output with any embedded address/data)
LEVEL 2: UNIT (subdivision like Gabinetto, Divisione II-A — output title only)
LEVEL 3: ROLE (Direttore, Capo Servizio — NOT a section, do not list)

Text:
"""

def detect_sections(raw_text):
    # detects sections, 'TITLE | field: value' format for propagation
    msg = [{"role": "user", "content": SECTION_DETECT_PROMPT + "\n" + raw_text}]
    res = rate_limited_invoke(llm,msg)
    result = clean(str(res.content))
    if "SINGLE_SECTION" in result:
        return None
    sections = []
    for line in result.split("\n"):
        line = line.strip()
        if not line:
            continue
        sections.append(line)
    return sections if sections else None


#section detection for Industria e commercio section
SECTION_DETECT_INDUSTRIA_PROMPT = """
Look at this text from a 1947 Venetian almanac page from the 'Industria e Commercio' section.
This section lists businesses organized by commercial category.
Category headings appear as short standalone lines (1-4 words, often italic or bold in the original).
Examples: "Camicerie da uomo", "Canapa e Lino", "Candele", "Cantieri Navali", "Agenzie di trasporto"
Neighborhood sub-headings also appear: "Lido", "Mestre", "Marghera", "Murano"

List each category heading and neighborhood heading found, one per line, in this format:
  CATEGORY_NAME | Category: CATEGORY_NAME
  NEIGHBORHOOD_NAME | Location: NEIGHBORHOOD_NAME

If no category or neighborhood headings are found, return: SINGLE_SECTION

Return only the list, nothing else.

Text:
"""

def detect_sections_industria(raw_text):
    # section detection for industria e commercio pages (132-318)
    msg = [{"role": "user", "content": SECTION_DETECT_INDUSTRIA_PROMPT + "\n" + raw_text}]
    res = rate_limited_invoke(llm,msg)
    result = clean(str(res.content))
    if "SINGLE_SECTION" in result:
        return None
    sections = []
    for line in result.split("\n"):
        line = line.strip()
        if not line:
            continue
        sections.append(line)
    return sections if sections else None


In [ ]:
LAYOUT_COMPLEXITY_PROMPT = """
Look at this OCR text from a historical document page.

Does the page contain visually distinct blocks such as:
- advertisements
- separated areas
- mixed layouts (not a simple continuous list)

Return ONLY:
- COMPLEX_LAYOUT
- SIMPLE_LAYOUT

Text:
"""

# Asks the LLM whether the page layout is simple or complex.
def detect_layout_complexity(raw_text):
    msg = [{"role": "user", "content": LAYOUT_COMPLEXITY_PROMPT + "\n" + raw_text}]
    res = rate_limited_invoke(llm,msg)
    return clean(str(res.content))


In [ ]:
# ADS

AD_DETECTION_PROMPT = """
Analyze this page.

Advertisements may include:
- visually separated text
- business promotions
- different formatting from the main list
- decorative or marketing-style blocks (not institutional headings)
- content at the bottom or edges of the page

Is this page:
- FULL_AD (entire page is ads)
- PARTIAL_AD (mix of directory content + ads, even small ones)
- NO_AD (pure structured directory content)

Be conservative:
If ANY advertisement-like content is present, return PARTIAL_AD.

Return only one label.

Text:
"""

# Checks the page number against hardcoded full-ad ranges.
def is_full_ad_page(page_num):
    if not page_num:
        return False
    ad_ranges = [
        (3, 5), (11, 13), (20, 21), (23, 23), (27, 27), (30, 31),
        (40, 41), (46, 47), (105, 107), (129, 131), (319, 331), (572, 575)
    ]
    for start, end in ad_ranges:
        if start <= page_num <= end:
            return True
    return False

AD_SEMANTIC_PROMPT = """

Extract entries into a CSV with columns:
Name,Address,Category,Additional Info

Definitions:
- Name: company or business name
- Address: address if present
- Category: type of business or profession (short, e.g. "Macchine per ufficio", "Cancelleria", "Officina riparazioni")
- Additional Info: any extra useful details

Rules:
- Do NOT classify institutional or governmental structures as advertisements
- Exclude: Dogana, Sezioni, Uffici, Circoscrizioni, Ispettorati, Ministerial offices
- Only extract explicit commercial promotional content (businesses, shops, services, printed ads)
- One row per business/entity (multiple rows allowed)
- If multiple activities are listed for the same business, group them in Category or Additional Info
- Use original language from the document (Italian). Do not translate any field into English.
- Do not invent data
- Leave fields empty if missing
- Keep Category concise
- Return ONLY CSV

Text:
"""

# Extracts ad entries into a Name/Address/Category/Additional Info CSV.
def detect_ads(raw_text):
    msg = [{"role": "user", "content": AD_DETECTION_PROMPT + "\n" + raw_text}]
    res = rate_limited_invoke(llm,msg)
    return clean(str(res.content))


In [ ]:
# INDEX PAGES

def is_index_page(page_type_hint):
    keywords = ["indice", "sommaire", "index", "elenco"]
    return any(k in page_type_hint.lower() for k in keywords)

INDEX_PROMPT = """
Extract a CSV with headers:
Category,Page

From this index/sommaire page.
If a category has a comma in its title, group everything under quotation marks (only if needed).
There must only be 2 fields.

Return ONLY CSV.

Text:
"""


In [ ]:
# PAGE TYPE INFERENCE

PAGE_TYPE_PROMPT = CONTEXT_HEADER + PAGE_TYPE_REFERENCE + """
Given the page number below, return in one short sentence what type of content this page likely contains,
based on the document structure above.
If the page is in a range that clearly has a consistent structure (like the alphabetical index 332-487),
say so briefly.
Return only that one sentence, nothing else.

Page number: {page_number}
"""

# Asks the LLM to classify the page type.
def infer_page_type(page_name):
    try:
        num = int(page_name.split("_")[-1])
    except ValueError:
        return ""
    prompt = PAGE_TYPE_PROMPT.format(page_number=num)
    msg = [{"role": "user", "content": prompt}]
    res = rate_limited_invoke(llm,msg)
    return clean(str(res.content))


In [ ]:
# PROVINCIA MODE (pages 494-570)

PROVINCIA_PROMPT = """
You are extracting general information from a section of a 1947 Venetian almanac.
This page describes one or more towns (provincie) in the Venice province.

For EACH provincia found on this page, extract ONLY the introductory block of text
that appears directly under the town name (in large uppercase letters).
This intro block contains geographic and demographic info: frazioni, abitanti, superficie, stazione, prodotti.
STOP extracting for that provincia when you reach ANY of these headers:
UFFICI AMMINISTRATIVI, UFFICI PUBBLICI, ISTITUZIONI, PROFESSIONISTI, COMMERCIO E INDUSTRIA

IMPORTANT: If a page starts mid-section (e.g. continuing a COMMERCIO or UFFICI block from the previous page)
with NO visible intro block for that provincia, do NOT output a row for it.
Only output a row if the intro block (demographic/geographic info) is visible on this page.

Return a CSV where each row is one provincia, with these columns:
Provincia, Frazioni, Abitanti, Superficie, Stazione, Prodotti, Altre_info

Definitions:
- Provincia: the large uppercase town name (e.g. ANNONE VENETO)
- Frazioni: list of frazioni dipendenti dal Comune, as one string
- Abitanti: number of inhabitants
- Superficie: surface area value
- Stazione: railway station info
- Prodotti: natural or industrial products
- Altre_info: any other general info in that intro block not covered above

Rules:
- If a field is not present, leave it empty
- Do NOT extract names, offices, roles, or business entries
- If multiple provincie appear on this page, output one row per provincia
- If only UFFICI/COMMERCIO/ISTITUZIONI content is visible with no intro block, return only the header row with no data rows
- Return ONLY CSV, no extra text
"""

PROVINCIA_AD_IMAGE_PROMPT = """
You are analyzing a scanned page from a 1947 Venetian almanac.
This page contains a directory listing of businesses (COMMERCIO E INDUSTRIA section).
It may also contain one or more ADVERTISEMENT BLOCKS — these are visually distinct from the main text:
they appear in bordered boxes, use larger or bolder fonts, or are clearly promotional in style.

Extract ONLY these advertisement blocks. Do NOT extract entries from the main directory listing.

Return a CSV with columns:
Name, Address, Category, Additional Info

Rules:
- Name: business or advertiser name
- Address: address if present in the ad
- Category: type of business (short)
- Additional Info: any slogan, phone number, or extra detail in the ad
- If no advertisement blocks are present, return only the header row
- Return ONLY CSV, no extra text
"""

PROVINCIA_PEOPLE_PROMPT = """
You are extracting structured data from a page of a 1947 Venetian almanac (Provincia di Venezia section).

The page lists one or more towns. Under each town, there are sections:
UFFICI AMMINISTRATIVI, UFFICI PUBBLICI, ISTITUZIONI, ISTITUZIONI DI BENEFICENZA,
PROFESSIONISTI, COMMERCIO E INDUSTRIA.

Your task: extract EVERY individual person and office/institution mentioned on this page.

Rules:
- One row per person or office per role/profession
- If multiple names share the same role (e.g. "Farmacisti: Rossi, Bianchi"), output one row per name
- Section = the section heading (e.g. UFFICI AMMINISTRATIVI, PROFESSIONISTI, COMMERCIO E INDUSTRIA)
- Role_or_Profession = the specific role or profession label before the colon (e.g. "Sindaco", "Farmacisti", "Barbieri")
- Name = the individual person's full name as written, including title if present (dott., avv., ing.)
- Address = the street address or building name explicitly stated for this person or office, if any. Leave empty if not stated.
- Provincia = the town name in uppercase (e.g. ANNONE VENETO)
- UFFICI PUBBLICI: extract the office name as Name, its address if present, section = UFFICI PUBBLICI
- Do NOT extract place names or general location names as Names
- Leave fields empty if genuinely missing

Columns: Provincia, Section, Role_or_Profession, Name, Address

CRITICAL CSV RULES:
- Any field containing a comma MUST be wrapped in double quotes.
  Example: "Agenzia compravend. immobili, affittanze, mediazioni"
  Example: "Biade, coloniali ed affini (Vendite e rappresentanze)"
  Example address: "Punta Sabbioni, Griolera, Caorle"
- This applies especially to Role_or_Profession and Address fields.
- Never split a single field across multiple columns due to internal commas.

Return ONLY CSV, no extra text.
"""


## Section-header propagation
Forward-fills addresses and section state across rows (Appendix~\ref{app:propagation}).


In [ ]:
#if the current row has a value in Address but it doesn't look like a real address, AND we have an inherited address from the section state, prefer the inherited one.

# Sestiere/parish keywords that indicate a row's Location is genuinely in Venice. 
VENICE_LOCATION_KEYWORDS = (
    "san marco", "s. marco", "s marco", "san polo", "s. polo", "s polo",
    "cannaregio", "cannareggio", "dorsoduro", "d. duro", "d.duro",
    "castello", "giudecca", "santa croce", "s. croce", "s croce",
    "sant'elena", "s. elena", "rialto", "venezia",
)


def _looks_like_venice_location(loc: str) -> bool:
    if not loc or not loc.strip():
        return True  # no Location to contradict the propagated address
    t = loc.strip().lower()
    return any(kw in t for kw in VENICE_LOCATION_KEYWORDS)


def forward_fill_sections(csv_text: str, inherited_state: dict = None) -> tuple[str, dict]:
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if not lines:
        return csv_text, inherited_state or {}

    reader = csv.reader(lines)
    rows = list(reader)
    if not rows:
        return csv_text, inherited_state or {}

    header = rows[0]
    current_section = dict(inherited_state) if inherited_state else {}

    # Best known real address — survives column name changes across pages
    best_address = current_section.get("__best_address__", "")

    def get_col(name):
        for h in header:
            if h.strip().lower() == name.lower():
                return h
        return None

    addr_col = get_col("Address")
    loc_col = get_col("Location")
    ai_col = get_col("Additional Info")

    output_rows = [header]

    for row in rows[1:]:
        if not row:
            continue

        row_dict = dict(zip(header, row + [""] * (len(header) - len(row))))
        name_val = row_dict.get("Name", "").strip()

        # --- __SECTION__ rows: update current_section and best_address ---
        if name_val == "__SECTION__":
            for col in header:
                if col == "Name":
                    continue
                val = row_dict.get(col, "").strip()
                if not val:
                    continue
                # Strip accidental "FieldName: " prefixes the LLM sometimes outputs
                for prefix in [f"{col}: ", f"{col.lower()}: ", "Address: ", "address: "]:
                    if val.startswith(prefix):
                        val = val[len(prefix):].strip()
                        break
                if col == addr_col:
                    if looks_like_address(val):
                        current_section[col] = val
                        best_address = val
                else:
                    # Don't propagate long promotional/ad text as section state
                    # Ad content: very long strings, or contains tel/telefono/servizi
                    is_ad_content = (
                        len(val) > 80
                        or any(kw in val.lower() for kw in [
                            "tel.", "telefono", "servizi rapidi", 
                            "negozi:", "corrispondenti:"
                        ])
                    )
                    if not is_ad_content:
                        current_section[col] = val
            continue

        # --- Normal rows ---
        row_dict_out = dict(row_dict)

        # --- Strip accidental "FieldName: " prefixes from all fields in normal rows ---
        for col in header:
            val = row_dict_out.get(col, "")
            if not val:
                continue
            for prefix in ["Address: ", "address: ", "Location: ", "location: ",
                           "Additional Info: ", "additional info: ", "Role: ", "role: "]:
                if val.strip().startswith(prefix):
                    stripped = val.strip()[len(prefix):].strip()
                    # Only strip if result is non-empty and makes sense
                    if stripped:
                        row_dict_out[col] = stripped
                    break
            # Also strip leading ", " artifacts
            val2 = row_dict_out.get(col, "")
            if val2.startswith(", "):
                row_dict_out[col] = val2[2:].strip()

        if addr_col:
            current_val = row_dict_out.get(addr_col, "").strip()
            had_own_address = bool(current_val)  # before any best_address injection below

            if not current_val:
                # Empty: use best known address, UNLESS this row's Location clearly names a non-Venice place 
                loc_val_for_check = row_dict_out.get(loc_col, "") if loc_col else ""
                if _looks_like_venice_location(loc_val_for_check):
                    row_dict_out[addr_col] = best_address
                # else: leave Address empty 

            elif re.match(r'^abitanti[:\s]', current_val, re.IGNORECASE):
                # Demographic data leaked into Address — move to Additional Info
                if ai_col:
                    existing = row_dict_out.get(ai_col, "").strip()
                    row_dict_out[ai_col] = (existing + " | " + current_val).strip(" |") if existing else current_val
                row_dict_out[addr_col] = best_address

            elif looks_like_address(current_val):
                # Good address: use it, update best_address
                best_address = current_val
                current_section[addr_col] = current_val

            else:
                # Bad value in address column: rescue it, inject best_address
                rescued = current_val
                row_dict_out[addr_col] = best_address  # always inject best address

                # Place rescued value in appropriate column
                if looks_like_sublocation(rescued):
                    if loc_col and not row_dict_out.get(loc_col, "").strip():
                        row_dict_out[loc_col] = rescued
                    elif ai_col:
                        existing = row_dict_out.get(ai_col, "").strip()
                        row_dict_out[ai_col] = (existing + " | " + rescued).strip(" |") if existing else rescued
                else:
                    if ai_col:
                        existing = row_dict_out.get(ai_col, "").strip()
                        row_dict_out[ai_col] = (existing + " | " + rescued).strip(" |") if existing else rescued
        
        # --- Location column cleanup ---
        # If Location contains an address-like value (has numbers),
        # it belongs in Address, not Location
        if loc_col:
            loc_val = row_dict_out.get(loc_col, "").strip()
            if loc_val and looks_like_address(loc_val) and re.search(r'\d', loc_val):
                # This is an address fragment, not a location name
                if addr_col:
                    current_addr = row_dict_out.get(addr_col, "").strip()
                    if had_own_address and current_addr and current_addr != loc_val:
                        # Row genuinely has two distinct address parts 
                        row_dict_out[addr_col] = current_addr + ", " + loc_val
                    else:
                        # Row had no address of its own 
                        row_dict_out[addr_col] = loc_val
                        best_address = loc_val
                        current_section[addr_col] = loc_val
                # Clear Location — it was an address fragment
                row_dict_out[loc_col] = ""

        # If Address is empty after all processing, check other columns for values that look like addresses 
        if addr_col and not row_dict_out.get(addr_col, "").strip():
            for col in header:
                if col in ("Name", addr_col, "page"):
                    continue
                col_val = row_dict_out.get(col, "").strip()
                if col_val and looks_like_address(col_val) and not re.search(r'\d', col_val):
                    # Looks like a church/location name, not a numbered street address
                    # Move to Address only if it's clearly a church signal
                    col_lower = col_val.lower()
                    is_church = any(re.search(r'(?<![a-z])' + re.escape(sig) + r'(?![a-z])', col_lower) 
                                   for sig in CHURCH_SIGNALS)
                    if is_church:
                        row_dict_out[addr_col] = col_val
                        best_address = col_val
                        current_section[addr_col] = col_val
                        row_dict_out[col] = ""  # clear from wrong column
                        break

        # Fill other empty columns from current_section
        for col in header:
            if col in ("Name", addr_col):
                continue
            if not row_dict_out.get(col, "").strip() and col in current_section:
                inherited_val = current_section[col]
                # Don't inherit Location values that contain numbers (address fragments)
                if col == loc_col and re.search(r'\d', inherited_val):
                    continue
                row_dict_out[col] = inherited_val

        output_rows.append([row_dict_out.get(col, "") for col in header])

    # Persist best_address into returned state so next page inherits it
    current_section["__best_address__"] = best_address

    output = StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    for row in output_rows:
        writer.writerow(row)

    return output.getvalue().strip(), current_section


## Provincia & data-quality checks
Header repair, truncated-name / missing-location warnings, address matching, church-row promotion.


In [ ]:
EXPECTED_PROVINCIA_COLUMNS = ["Provincia", "Section", "Role_or_Profession", "Name", "Address"]

def ensure_provincia_header(csv_text: str) -> str:
    # prepends the header if the LLM output a data row as the first line
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if not lines:
        return csv_text

    reader = csv.reader(lines)
    rows = list(reader)
    if not rows:
        return csv_text

    first_row = [c.strip() for c in rows[0]]
    expected_lower = [c.lower() for c in EXPECTED_PROVINCIA_COLUMNS]

    # Check how many expected column names appear in the first row
    matches = sum(1 for c in first_row if c.lower() in expected_lower)

    if matches >= 3:
        # First row looks like a real header — fine as-is
        return csv_text

    # First row looks like data — prepend the correct header
    output = StringIO()
    writer = csv.writer(output)
    writer.writerow(EXPECTED_PROVINCIA_COLUMNS + ["page"])
    for row in rows:
        n = len(EXPECTED_PROVINCIA_COLUMNS) + 1
        if len(row) < n:
            row = row + [""] * (n - len(row))
        writer.writerow(row[:n])
    return output.getvalue().strip()


In [ ]:
# Regex for name-only rows missing an address/title
TITLE_ONLY_PATTERNS = re.compile(
    r'^(dott|rag|ing|prof|avv|cav|dr|sig|comm|geom)\.?$',
    re.IGNORECASE
)

def warn_truncated_names(csv_text: str, page_name: str):
    # warns if a Name field looks like a truncated title
    logger = logging.getLogger("ocr_pipeline")
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if len(lines) < 2:
        return
    reader = csv.reader(lines)
    rows = list(reader)
    if not rows:
        return
    header = rows[0]
    try:
        name_idx = [h.strip().lower() for h in header].index("name")
    except ValueError:
        return
    for i, row in enumerate(rows[1:], start=2):
        if len(row) > name_idx:
            name_val = row[name_idx].strip()
            if TITLE_ONLY_PATTERNS.match(name_val):
                logger.info(f"  -> WARNING: truncated name '{name_val}' on row {i} of {page_name}")


def warn_missing_location(csv_text: str, page_name: str):
    # warns if a row has both Address and Location empty
    logger = logging.getLogger("ocr_pipeline")
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if len(lines) < 2:
        return
    reader = csv.reader(lines)
    rows = list(reader)
    if not rows:
        return
    header = rows[0]
    header_lower = [h.strip().lower() for h in header]

    addr_idx = next((i for i, h in enumerate(header_lower) if h == "address"), None)
    loc_idx = next((i for i, h in enumerate(header_lower) if h == "location"), None)

    if addr_idx is None and loc_idx is None:
        return

    missing_count = 0
    for row in rows[1:]:
        addr_val = row[addr_idx].strip() if addr_idx is not None and len(row) > addr_idx else ""
        loc_val = row[loc_idx].strip() if loc_idx is not None and len(row) > loc_idx else ""
        if not addr_val and not loc_val:
            missing_count += 1

    if missing_count > 0:
        pct = round(100 * missing_count / max(len(rows) - 1, 1))
        logger.info(f"  -> WARNING: {missing_count} rows ({pct}%) missing both Address and Location on {page_name}")


In [ ]:
# Keywords that mark a field as an address
VENETIAN_ADDRESS_SIGNALS = [
    "calle", "campo", "fondamenta", "riva", 
    "sestiere", "palazzo", "corte", "via", "piazza",
    "ponte", "dorsoduro", "cannaregio", "castello", "s. marco", "san marco",
    "s. polo", "san polo", "s. croce", "santa croce", "giudecca", "lido",
     "piazzale", "istituti", "istituto",
    "accademia", "rialto", "prigioni", "stae", "s. stae",
    "s. angelo", "san angelo", "s. silvestro", "san silvestro", "numero",
    "s. zulian", "san zulian", "s. marta", "san marta", "mirano", "chioggia", "dolo", "mestre", "caorle", "murano",
    "s. maria", "san", "santa", "ss.", "chiesa", "parrocchia",
    "convento", "oratorio", "basilica", "cattedrale", "cappella"
]

ADDRESS_BLOCKLIST = {
    "capo", "direttore", "segretario", "presidente", "membro",
    "titolare", "procuratore", "ispettore", "commissario",
    "dirigente", "gestore", "ricevitore", "abitanti", "superficie", "stazione", "consigliere"
}

CHURCH_SIGNALS = [
    "basilica", "parrocchia", "chiesa", "convento", "oratorio",
    "s. marco", "s. apostoli", "s. canciano", "s. giovanni", "s. pietro",
    "s. maria", "san ", "santa ", "ss.", "s. ", "diocesano"
]

# Heuristic address match against Venetian/church signal words.
def looks_like_address(value: str) -> bool:
    if not value:
        return False
    v = value.strip()

    if v.lower() in ADDRESS_BLOCKLIST:
        return False

    if v.isupper() and len(v) > 15:
        v_lower = v.lower()
        has_church = any(sig in v_lower for sig in CHURCH_SIGNALS)
        has_venetian = any(sig in v_lower for sig in VENETIAN_ADDRESS_SIGNALS)
        if not has_church and not has_venetian:
            return False

    if len(v) < 8 and not re.search(r'\d', v):
        return False

    if re.search(r'\d', v):
        return True

    v_lower = v.lower()
    for sig in VENETIAN_ADDRESS_SIGNALS:
        pattern = r'(?<![a-z])' + re.escape(sig) + r'(?![a-z])'
        if re.search(pattern, v_lower):
            return True
    for sig in CHURCH_SIGNALS:
        pattern = r'(?<![a-z])' + re.escape(sig) + r'(?![a-z])'
        if re.search(pattern, v_lower):
            return True

    return False

SUBLOCATION_SIGNALS = [
    "sezione", "divisione", "ufficio", "uffici", "reparto",
    "compartimentale", "circoscrizione", "ricevitoria", "manifattura",
    "magazzino", "commissione", "deposito", 
    "finanziari di", "dogana secondaria", "dogana principale",
]

def looks_like_sublocation(value: str) -> bool:
    # True if value looks like an institutional sub-location, not a street address
    if not value:
        return False
    v_lower = value.strip().lower()
    return any(sig in v_lower for sig in SUBLOCATION_SIGNALS)


In [ ]:
def promote_church_rows_to_sections(csv_text: str) -> str:
    # converts church/location Name rows to __SECTION__ rows for propagation
    lines = [l for l in csv_text.split("\n") if l.strip()]
    if not lines:
        return csv_text

    reader = csv.reader(lines)
    rows = list(reader)
    if not rows:
        return csv_text

    header = rows[0]
    header_lower = [h.strip().lower() for h in header]

    def get_idx(name):
        try:
            return header_lower.index(name.lower())
        except ValueError:
            return None

    name_idx = get_idx("Name")
    addr_idx = get_idx("Address")
    role_idx = get_idx("Role")
    ai_idx = get_idx("Additional Info")

    if name_idx is None:
        return csv_text

    PERSON_SIGNALS = re.compile(
        r'\b(don|mons|padre|p\.|dott|dr|rag|ing|prof|avv|cav|sig|comm|geom|'
        r'rev|ecc|em|ill)\b', re.IGNORECASE
    )

    ORGANIZATIONAL_ROLES = {
        "ufficio missionario diocesano", "esaminatori prosinodali",
        "consultori dei parroci", "censori dei libri",
        "commissione", "tribunale", "patronato", "ufficio"
    }

    output = StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header)

    for row in rows[1:]:
        if not row:
            continue
        if len(row) < len(header):
            row = row + [""] * (len(header) - len(row))

        name_val = row[name_idx].strip() if name_idx < len(row) else ""
        role_val = row[role_idx].strip().lower() if role_idx is not None and role_idx < len(row) else ""

        # Already a section row
        if name_val == "__SECTION__":
            writer.writerow(row)
            continue

        # Check if this row is a church/location name, not a person
        is_person = bool(PERSON_SIGNALS.search(name_val))
        is_org_role = any(kw in role_val for kw in ORGANIZATIONAL_ROLES)
        name_lower = name_val.lower()
        is_church_name = (
            any(re.search(r'(?<![a-z])' + re.escape(sig) + r'(?![a-z])', name_lower)
                for sig in CHURCH_SIGNALS)
            or (name_val.isupper() and len(name_val) > 5)
        )

        if is_church_name and not is_person and (not role_val or is_org_role):
            # Convert to __SECTION__ row: church name → Address
            new_row = [""] * len(header)
            new_row[name_idx] = "__SECTION__"
            if addr_idx is not None:
                new_row[addr_idx] = name_val
            # Move any additional info to ai column
            if ai_idx is not None and ai_idx < len(row):
                ai_val = row[ai_idx].strip()
                if ai_val and not any(kw in ai_val.lower() for kw in ["commissione", "ufficio"]):
                    new_row[ai_idx] = ""  # drop building descriptions
            writer.writerow(new_row)
        else:
            writer.writerow(row)

    return output.getvalue().strip()


## Main extraction loop
CSV validation, the retrying LLM-call wrapper, and the per-page pipeline.


In [ ]:
# CSV VALIDATION + SEMANTIC RUNNER

def is_valid_csv(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if len(lines) < 2:
        return False
    if "," not in lines[0]:
        return False
    if not any("," in l for l in lines[1:]):
        return False
    return True


In [ ]:
# Wraps a prompt and base64 image into the LLM message format.
def build_image_message(text, b64):
    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": text},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}", "detail": "high"}}
        ]
    }]


def run_semantic(prompt, raw_text=None, b64=None, mode="text", max_retries=2):
    # text or image message to the LLM, retries with simpler prompts
    logger = logging.getLogger("ocr_pipeline")
    def call_llm(p):
        if mode == "text":
            msg = [{"role": "user", "content": p + "\n" + raw_text}]
        else:
            msg = build_image_message(p, b64)
        res = rate_limited_invoke(llm,msg)
        return clean(str(res.content))

    result = call_llm(prompt)

    for attempt in range(max_retries + 1):
        if not is_refusal(result) and is_valid_csv(result):
            return result

        logger.info(f"  -> Invalid semantic output (attempt {attempt+1}), retrying...")

        if attempt == 0:
            retry_prompt = prompt + """

IMPORTANT:
- You MUST return a valid CSV
- At least 2 rows (header + one entry)
- Do NOT explain anything
- Do NOT say "no data"
- If unsure, extract best possible rows
"""
        elif attempt == 1:
            retry_prompt = FALLBACK_PROMPT
        else:
            retry_prompt = """
Extract ANY structured entries from this text.

Return strictly:
Name,Address,Notes

At least one row must be present.
Do not output anything else.
"""
        result = call_llm(retry_prompt)

    return result


In [ ]:
# MAIN PIPELINE

PAGES_DIR = Path(PAGES_DIR)


# Main per-page loop: OCR, section detection, semantic extraction, propagation, save.
def run_ocr(page_files, progress_file: Path = None,
            drive_service=None,
            drive_root_folder_id: str = None):
    # main pipeline over (page_name, drive_file_id) tuples

    logger = logging.getLogger("ocr_pipeline")

    def mark_done():
        if progress_file:
            mark_progress(progress_file, page_name)

    # Strip extensions from page names 
    # Drive image lookup uses file_id directly so stripping the key name is safe.
    page_names = [re.sub(r'\.[a-zA-Z0-9]+$', '', name) for name, _ in page_files]
    page_ids   = {
        re.sub(r'\.[a-zA-Z0-9]+$', '', name): fid
        for name, fid in page_files
    }

    seen = set()
    page_names = [n for n in page_names if not (n in seen or seen.add(n))]


    out_dir = Path(OCR_RESULTS_DIR)
    if SAVE_MODE in ("local", "both"):
        out_dir.mkdir(parents=True, exist_ok=True)

    # Load progress for resume support
    completed_pages = load_progress(progress_file) if progress_file else set()

    # Initialize state
    section_state = {}
    provincia_rows = {}
    provincia_ads = []
    last_provincia_name = None
    section_schema = None

    if completed_pages:
        logger.info(f"  -> Resuming: {len(completed_pages)} pages already done, rebuilding state...")

        completed_nums = sorted(
            extract_page_number(p)
            for p in completed_pages
            if extract_page_number(p)
        )

        if completed_nums:
            last_page = completed_nums[-1]
            last_page_name = f"page_{last_page}"

            page_dir = out_dir / last_page_name
            csv_files = sorted(page_dir.glob("*_semantic_*.csv"))

            if csv_files:
                last_csv = csv_files[-1]
                csv_text = last_csv.read_text(encoding="utf-8")

                if last_page not in DOCUMENT_SECTION_BOUNDARIES:
                    _, section_state = forward_fill_sections(csv_text)
                    section_schema = csv_text.splitlines()[0]

                logger.info(f"  -> Restored propagation state from {last_page_name}")
            else:
                logger.info(f"  -> No semantic CSV found for {last_page_name}, starting fresh.")

    # Loop over every page
    for page_name in page_names:
        file_id = page_ids[page_name]

        if page_name in completed_pages:
            continue

        try:
            fh = io.BytesIO(download_bytes(drive_service, file_id))
            img = Image.open(fh).convert("RGB")

            logger.info(f"Processing: {page_name}")
            page_num = extract_page_number(page_name)
            b64 = encode_image(img)

            # Compute run_id ONCE per page, before any branching or LLM calls.
            # All save() calls below reuse this same run_id.
            page_dir = out_dir / page_name
            if SAVE_MODE in ("local", "both"):
                page_dir.mkdir(parents=True, exist_ok=True)

            page_folder_id = None
            if SAVE_MODE in ("drive", "both") and drive_service and drive_root_folder_id:
         
                page_folder_id = get_or_create_drive_folder(
                    drive_service, page_name, drive_root_folder_id
                )

            run_id = get_next_run_id(page_dir, page_name,
                                    drive_service=drive_service,
                                    drive_folder_id=page_folder_id)

            # ---- STEP 1: IMAGE → TEXT (OCR) ----
            reading_msg = build_image_message(READING_PROMPT, b64)

            res_text = rate_limited_invoke(llm, reading_msg)
            raw_text = clean(str(res_text.content))

            if is_refusal(raw_text):
                logger.info("  -> OCR refusal detected, retrying with fallback prompt...")
                fallback_msg = build_image_message("Transcribe all visible text exactly. Do not refuse.", b64)
                res_text = rate_limited_invoke(llm, fallback_msg)
                raw_text = clean(str(res_text.content))
                if is_refusal(raw_text):
                    logger.info(f"  -> OCR still refused after fallback for {page_name}, skipping.")
                    mark_done()
                    continue

            cleaned_text = raw_text

            if len(cleaned_text.strip()) < 50:
                logger.info(f"  -> Page appears blank or near-empty, skipping semantic processing.")
                save(page_dir / f"{page_name}_ocr_{run_id}.txt", raw_text,
     drive_service=drive_service, drive_folder_id=page_folder_id)
                mark_done()
                continue

            # Provincia mode (pages 494-570): skip normal extraction
            if PROVINCIA_MODE and page_num and 494 <= page_num <= 570:

                save(page_dir / f"{page_name}_ocr_{run_id}.txt", raw_text, drive_service=drive_service, drive_folder_id=page_folder_id)

                if is_full_ad_page(page_num):
                    mark_done()
                    continue

                ad_msg = build_image_message(PROVINCIA_AD_IMAGE_PROMPT, b64)
                ad_result = rate_limited_invoke(llm, ad_msg)
                ad_csv = clean(str(ad_result.content))
                ad_lines = [l for l in ad_csv.split("\n") if l.strip()]

                if len(ad_lines) >= 2:
                    ad_reader = csv.reader(ad_lines)
                    ad_rows_parsed = list(ad_reader)
                    for ad_row in ad_rows_parsed[1:]:
                        if ad_row and any(cell.strip() for cell in ad_row):
                            while len(ad_row) < 4:
                                ad_row.append("")
                            provincia_ads.append(ad_row[:4] + [page_num])
                    logger.info(f"  -> Provincia ads extracted for page {page_num}.")

                #csv extraction of the provincia info
                csv_text = run_semantic(PROVINCIA_PROMPT, raw_text=cleaned_text, mode="text")

                # --- Per-page people extraction ---

                # Build a provincia-aware prompt that carries the last known town forward
                if last_provincia_name:
                    provincia_context = (
                        f"\nIMPORTANT CONTEXT: This page is a continuation. "
                        f"The last known town name from the previous page was '{last_provincia_name}'. "
                        f"If this page has NO visible town heading in uppercase, use '{last_provincia_name}' "
                        f"as the Provincia value for all rows on this page.\n"
                    )
                else:
                    provincia_context = ""

                people_csv = run_semantic(
                    PROVINCIA_PEOPLE_PROMPT + provincia_context,
                    raw_text=cleaned_text,
                    mode="text"
                )
                people_csv = ensure_provincia_header(people_csv)

                # Backfill empty Provincia values using last_provincia_name
                if last_provincia_name and people_csv:
                    people_lines = [l for l in people_csv.split("\n") if l.strip()]
                    if len(people_lines) >= 2:
                        reader = csv.reader(people_lines)
                        rows = list(reader)
                        header = rows[0]
                        header_lower = [h.strip().lower() for h in header]
                        try:
                            prov_idx = header_lower.index("provincia")
                            output = StringIO()
                            writer = csv.writer(output)
                            writer.writerow(header)
                            for row in rows[1:]:
                                if len(row) <= prov_idx or not row[prov_idx].strip():
                                    while len(row) <= prov_idx:
                                        row.append("")
                                    row[prov_idx] = last_provincia_name
                                writer.writerow(row)
                            people_csv = output.getvalue().strip()
                        except ValueError:
                            pass  # no Provincia column — skip backfill

                people_csv = add_page_column(people_csv, page_num)
                people_lines = [l for l in people_csv.split("\n") if l.strip()]
                if len(people_lines) >= 2:
                    save(page_dir / f"{page_name}_semantic_{run_id}.csv", people_csv,
                        drive_service=drive_service, drive_folder_id=page_folder_id)
                    n_people = len(people_lines) - 1
                    logger.info(f"  -> Semantic CSV saved: {n_people} entries for page {page_num}.")

                # Update last_provincia_name from the people CSV if a new town was found
                if people_csv:
                    _p_lines = [l for l in people_csv.split("\n") if l.strip()]
                    if len(_p_lines) >= 2:
                        _reader = csv.reader(_p_lines)
                        _rows = list(_reader)
                        _header_lower = [h.strip().lower() for h in _rows[0]]
                        try:
                            _pidx = _header_lower.index("provincia")
                            for _row in _rows[1:]:
                                if len(_row) > _pidx and _row[_pidx].strip():
                                    _candidate = _row[_pidx].strip()
                                    # Only update if it's all-caps (real town name, not backfilled)
                                    if _candidate.isupper() and _candidate != last_provincia_name:
                                        last_provincia_name = _candidate
                                        break
                        except ValueError:
                            pass

                lines = [l for l in csv_text.split("\n") if l.strip()]
                found_new_provincia = False

                if len(lines) >= 2:
                    reader = csv.reader(lines)
                    prov_rows = list(reader)
                    prov_header = prov_rows[0]

                    for prov_row in prov_rows[1:]:
                        if len(prov_row) < len(prov_header):
                            prov_row += [""] * (len(prov_header) - len(prov_row))

                        row_data = dict(zip(prov_header, prov_row))
                        prov_name = row_data.get("Provincia", "").strip()

                        if not prov_name:
                            continue

                        found_new_provincia = True

                        if prov_name in provincia_rows:
                            for k, v in row_data.items():
                                if v and not provincia_rows[prov_name]["data"].get(k):
                                    provincia_rows[prov_name]["data"][k] = v
                            if page_num not in provincia_rows[prov_name]["pages"]:
                                provincia_rows[prov_name]["pages"].append(page_num)
                        else:
                            provincia_rows[prov_name] = {
                                "data": row_data,
                                "pages": [page_num]
                            }
                        last_provincia_name = prov_name

                    logger.info(f"  -> Provincia data extracted for page {page_num}.")

                if not found_new_provincia:
                    if last_provincia_name and last_provincia_name in provincia_rows:
                        if page_num not in provincia_rows[last_provincia_name]["pages"]:
                            provincia_rows[last_provincia_name]["pages"].append(page_num)
                        logger.info(f"  -> Page {page_num} added to '{last_provincia_name}' (continuation).")

                mark_done()
                    
                continue

            # ---- AD DETECTION ----
            layout_type = detect_layout_complexity(raw_text)
            logger.info(f"  -> Layout type: {layout_type}")

            if is_full_ad_page(page_num):
                ad_type = "FULL_AD"
            else:
                ad_type = detect_ads(raw_text) if layout_type == "COMPLEX_LAYOUT" else "NO_AD"

            logger.info(f"  -> Ad type: {ad_type}")

            # ---- FULL AD PAGES ----
            if ad_type == "FULL_AD":
                save(page_dir / f"{page_name}_ocr_{run_id}.txt", raw_text, drive_service=drive_service, drive_folder_id=page_folder_id)
                csv_text = run_semantic(AD_SEMANTIC_PROMPT, raw_text=raw_text, mode="text")
                first_line = [l for l in csv_text.split("\n") if l.strip()][0]
                n_cols = len(next(csv.reader([first_line])))
                csv_text = repair_csv(csv_text, expected_columns=n_cols)
                csv_text, _ = forward_fill_sections(csv_text, inherited_state={})
                csv_text = add_page_column(csv_text, page_num)
                save(page_dir / f"{page_name}_semantic_{run_id}.csv", csv_text, drive_service=drive_service, drive_folder_id=page_folder_id)
                mark_done()
                continue

            # ----  TEXT → CSV ----

            page_type_hint = infer_page_type(page_name)
            logger.info(f"  -> Page type inferred: {page_type_hint}")

            is_idx = (
                (is_index_page(page_type_hint) and not (page_num and 332 <= page_num <= 487))
                or (page_num and 14 <= page_num <= 35) or (page_num == 10)
            )
            if is_idx:
                logger.info(f"  -> Index/sommaire page detected, extracting...")
                csv_text = run_semantic(INDEX_PROMPT, raw_text=cleaned_text, mode="text")
                save(page_dir / f"{page_name}_ocr_{run_id}.txt", raw_text,
                    drive_service=drive_service, drive_folder_id=page_folder_id)
                first_line = [l for l in csv_text.split("\n") if l.strip()][0]
                n_cols = len(next(csv.reader([first_line])))
                csv_text = repair_csv(csv_text, expected_columns=n_cols)
                csv_text, _ = forward_fill_sections(csv_text, inherited_state={})
                csv_text = add_page_column(csv_text, page_num)

            
                save(page_dir / f"{page_name}_semantic_{run_id}.csv", csv_text,
                     drive_service=drive_service, drive_folder_id=page_folder_id)
                mark_done()
                logger.info(f"  -> Index CSV saved ({len([l for l in csv_text.splitlines() if l.strip()])-1} rows). Done.")
                continue

            if section_state:
                inherited_columns = ", ".join(
                    k for k in section_state.keys() if not k.startswith("__")
                )
                schema_hint = (
                    f"This page continues a section. "
                    f"Prefer these columns from the previous page: {inherited_columns}. "
                    f"Only add new columns if the page clearly requires them."
                )
            else:
                schema_hint = ""

            if page_num and 132 <= page_num <= 318:
                section_type_hint = (
                    "This page is from the 'Industria e Commercio' section (pages 132-318). "
                    "Entries are businesses organized by commercial category, "
                    "then sometimes by neighborhood (Lido, Mestre, Marghera). "
                    "Preferred columns: Name, Address, Location, Category, Additional Info. "
                    "No Role or Profession columns needed in this section."
                )
            elif page_num and 108 <= page_num <= 128:
                section_type_hint = (
                    "This page is from the 'Professionisti e Artisti' section (pages 108-128). "
                    "Entries are professionals listed alphabetically by profession category. "
                    "Preferred columns: Name, Address, Location, Profession, Additional Info."
                )
            elif page_num and (86 <= page_num <= 97 or 36 <= page_num <= 37):
                section_type_hint = (
                    "This page is from the 'Istituzioni Religiose' section (pages 86-97 and pages 36-37). "
                    "Preferred columns: Name, Address (name of the church), Location (city or province), Role (ex: Vescovo, Arc...), Additional Info."
                )
            elif page_num and 332 <= page_num <= 487:
                section_type_hint = (
                    "This page is from the 'Indice Generale Alfabetico' section (pages 332-487). "
                    "Preferred columns: Name, Address, Location, Category, Additional Info."
                )
            else:
                section_type_hint = ""

            inferred_schema = infer_schema(
                (section_type_hint + "\n\n" if section_type_hint else "") + cleaned_text
            )
            logger.info(f"  -> Inferred schema: {inferred_schema}")

            if page_num in DOCUMENT_SECTION_BOUNDARIES:
                section_schema = None

            if section_schema is None:
                section_schema = inferred_schema
                schema = inferred_schema
            else:
                schema = merge_schemas(section_schema, inferred_schema)
                section_schema = schema

            logger.info(f"  -> Final schema: {schema}")

            if ad_type == "PARTIAL_AD":
                ad_hint = (
                    "- This page contains advertisements mixed with directory entries. "
                    "For ad entries: Name=business name, Address=address if present. "
                    f"Use existing columns ({schema}) for all other data. "
                    "Put the type of business (e.g. 'Arti Grafiche', 'Officina Riparazioni') in Additional Info.\n"
                    "Leave Role, Profession empty for ad rows.\n"
                    "Do NOT put descriptions in the Address field.\n"
                )
            else:
                ad_hint = ""

            provincia_keywords = ["provincia", "comune", "towns", "locality", "localities", "municipal"]
            if any(k in page_type_hint.lower() for k in provincia_keywords):
                address_hint = (
                    "- Split addresses into two fields: "
                    "Address (street or building name and number) and City (town or comune). "
                    "Leave City empty if not present.\n"
                )
            else:
                address_hint = (
                    "- Split addresses into two fields only when both parts are present:\n"
                    "  Address = the specific street, calle, campo, palazzo, convento, or building WITH its number if present.\n"
                    "  Location = ONLY a sestiere name (Dorsoduro, Cannaregio, Castello, S. Marco, S. Polo, S. Croce, Giudecca) "
                    "or a city/neighborhood name with NO street number.\n"
                    "- If the address has two parts like 'Palazzo Papadopoli, S. Silvestro n. 1364', "
                    "keep the FULL address in Address and leave Location empty.\n"
                    "- Never put a value with a street number in Location.\n"
                    "- Never put an institutional name (Ufficio, Sezione, Divisione, Ispettorato) in Address or Location.\n"
                    "- Leave Location empty if no clean sestiere or city name is present.\n"
                )

            prompt = (
                CONTEXT_HEADER
                + f"\nPage type context: {page_type_hint}\n"
                + f"You MUST use exactly these columns: {schema}\n"
                + (schema_hint + "\n" if schema_hint else "")
                + "The Name column must contain the person's full name. Roles, titles, and positions go in their own column.\n"
                + "Only add 'Additional Info' if there is genuinely no better-named column for the data.\n"
                + address_hint
                + ad_hint
                + "\n"
                + SEMANTIC_PROMPT_FROM_TEXT
            )

            if page_num and 132 <= page_num <= 318:
                sections = detect_sections_industria(cleaned_text)
            elif page_num and (86 <= page_num <= 97 or 36 <= page_num <= 37):
                sections = detect_sections(cleaned_text)
                if sections:
                    reformatted = []
                    for s in sections:
                        if "|" not in s:
                            s_upper = s.upper()
                            if any(kw in s_upper for kw in [
                                "COMMISSIONE", "CONSIGLIO", "COMITATO",
                                "UFFICIO", "SEGRETERIA", "ESAMINATORI",
                                "CONSULTORI", "CENSORI", "CANONICI", "CAPPELLANI"
                            ]):
                                reformatted.append(f"{s} | Additional Info: {s}")
                            elif (
                                len(s.split()) <= 3
                                and not s_upper.startswith("SS.")
                                and not s_upper.startswith("S.")
                                and not s_upper.startswith("SAN ")
                                and not s_upper.startswith("SANTA ")
                                and not s_upper.startswith("CHIESA")
                                and not s_upper.startswith("BASILICA")
                                and not s_upper.startswith("PARR")
                                and s[0].isupper()
                                and not s.isupper()
                            ):
                                reformatted.append(f"{s} | Location: {s}")
                            else:
                                reformatted.append(f"{s} | Address: {s}")
                        else:
                            reformatted.append(s)
                    sections = reformatted
            else:
                sections = detect_sections(cleaned_text)

            if sections:
                sections = [
                    s for s in sections
                    if not any(x in s.upper() for x in [
                        "GUIDA COMMERCIALE DI VENEZIA",
                        "GU1DA COMMERCIALE DI VENEZIA",
                        "GUÌDA COMMERCIALE DI VENEZIA",
                        "GUDA COMMERCIALE DI VENEZIA",
                        "GU DA COMMERCIALE DI VENEZIA",
                        "GRIDA COMMERCIALE DI VENEZIA",
                        "COMMERCIO, INDUSTRIA, PROFESSIONISTI",
                        "COMMERCIO E INDUSTRIA",
                        "INDUSTRIA E COMMERCIO",
                    ])
                ]

            if sections:
                schema_cols = [c.strip() for c in schema.split(",")]
                for s in sections:
                    if "|" not in s:
                        continue
                    _, payload = s.split("|", 1)
                    for part in payload.split(","):
                        if ":" not in part:
                            continue
                        k, v = part.split(":", 1)
                        k = k.strip()
                        v = v.strip()
                        if not k or not v:
                            continue
                        if k in schema_cols:
                            section_state[k] = v
                            if k.lower() == "address":
                                section_state["__best_address__"] = v

            if sections:
                logger.info(f"  -> Multi-section page detected: {sections}")
                sections_text = "\n".join(f"- {s}" for s in sections)
                active_prompt = (
                    prompt
                    + "\n\nSECTION INFERENCE INSTRUCTIONS:\n"
                    + "The following sections were detected. Format is: TITLE | field: value (embedded data from the section heading).\n"
                    + "For each section: output a row with '__SECTION__' in the Name column.\n"
                    + "Fill the other columns using the embedded field:value pairs if present.\n"
                    + "If a section has an Address embedded, put it in the Address column of the __SECTION__ row.\n"
                    + "Then output all entries belonging to that section as normal rows below it.\n"
                    + "Sub-sections should also get __SECTION__ rows with their title in the appropriate column.\n"
                    + "Do NOT carry forward values yourself — just output __SECTION__ rows correctly.\n"
                    + "Detected sections:\n"
                    + sections_text
                    + "\n"
                )
            else:
                active_prompt = prompt

            use_image_repass = (
                layout_type == "COMPLEX_LAYOUT"
                and ad_type == "PARTIAL_AD"
                and sections is not None
                and len(sections) >= 3
            )

            if use_image_repass:
                logger.info(f"  -> Image re-pass enabled (complex layout + ads + {len(sections)} sections).")
                combined_text = (
                    active_prompt
                    + "\n\n--- ADDITIONAL VISUAL CONTEXT ---\n"
                    + "The image of this page is also provided.\n"
                    + "Use the image ONLY to resolve structural ambiguity.\n"
                    + "Use the OCR text below for exact spelling of all names and addresses.\n"
                    + "OCR TEXT:\n"
                    + cleaned_text
                )
                combined_msg = build_image_message(combined_text, b64)
                res = rate_limited_invoke(llm, combined_msg)
                csv_text = clean(str(res.content))
                if not is_valid_csv(csv_text) or is_refusal(csv_text):
                    logger.info(f"  -> Image re-pass failed, falling back to text-only.")
                    csv_text = run_semantic(active_prompt, raw_text=cleaned_text, mode="text")
            else:
                csv_text = run_semantic(active_prompt, raw_text=cleaned_text, mode="text")

            actual_header_lines = [l for l in csv_text.split("\n") if l.strip()]
            if actual_header_lines:
                logger.info(f"  -> Actual CSV columns: {actual_header_lines[0]}")

            # ---- SAVE ----
            save(page_dir / f"{page_name}_ocr_{run_id}.txt", raw_text,
     drive_service=drive_service, drive_folder_id=page_folder_id)

            first_line = [l for l in csv_text.split("\n") if l.strip()][0]
            n_cols = len(next(csv.reader([first_line])))
            csv_text = repair_csv(csv_text, expected_columns=n_cols)
            warn_truncated_names(csv_text, page_name)
            warn_missing_location(csv_text, page_name)

            if page_num and (36 <= page_num <= 37 or 86 <= page_num <= 97):
                csv_text = promote_church_rows_to_sections(csv_text)

            if page_num in DOCUMENT_SECTION_BOUNDARIES:
                section_state = {}
                section_schema = None
                logger.info(f"  -> Section state reset at document boundary (page {page_num})")

            csv_text, section_state = forward_fill_sections(csv_text, inherited_state=section_state)
            if section_state.get("__best_address__"):
                logger.info(f"  -> Section state carries address: {section_state['__best_address__']}")

            csv_text = add_page_column(csv_text, page_num)
            save(page_dir / f"{page_name}_semantic_{run_id}.csv", csv_text,
     drive_service=drive_service, drive_folder_id=page_folder_id)

            mark_done()
            logger.info("Done.")

        except Exception as e:
            logger.info(f"  -> ERROR on {page_name}: {e}, skipping.")
            continue

    # Flush remaining provincia/ads rows to CSV
    if PROVINCIA_MODE and (provincia_rows or provincia_ads):

        flush_out_dir = Path.cwd() / "llm_ocr_results"
        if SAVE_MODE in ("local", "both"):
            flush_out_dir.mkdir(exist_ok=True)

        v = None
        provincia_folder_id = None

        if provincia_rows:
            all_keys = []
            for entry in provincia_rows.values():
                for k in entry["data"].keys():
                    if k not in all_keys:
                        all_keys.append(k)
            all_keys.append("page")

            output = StringIO()
            writer = csv.writer(output)
            writer.writerow(all_keys)
            for prov_name, entry in provincia_rows.items():
                page_str = "-".join(map(str, sorted(entry["pages"])))
                row = [entry["data"].get(k, "") for k in all_keys[:-1]] + [page_str]
                writer.writerow(row)

            # Save to provincia/ subfolder under root — create it if it doesn't exist.
            # This keeps aggregate files separate from per-page results.
            provincia_folder_id = None
            if SAVE_MODE in ("drive", "both") and drive_service and drive_root_folder_id:
                provincia_folder_id = get_or_create_drive_folder(
                    drive_service, "provincia", drive_root_folder_id
                )

            v = get_next_provincia_version(
                "provincia_di_venezia", flush_out_dir,
                drive_service=drive_service, drive_folder_id=provincia_folder_id
            )
            save(flush_out_dir / f"provincia_di_venezia_{v}.csv", output.getvalue(),
                drive_service=drive_service, drive_folder_id=provincia_folder_id)
            logger.info(f"  -> Provincia CSV saved: {len(provincia_rows)} entries (version {v}).")

        if provincia_ads:
            ads_output = StringIO()
            ads_writer = csv.writer(ads_output)
            ads_writer.writerow(["Name", "Address", "Category", "Additional Info", "page"])
            for ad_row in provincia_ads:
                while len(ad_row) < 5:
                    ad_row.append("")
                ads_writer.writerow(ad_row[:5])

            # Reuse provincia_folder_id from above if provincia_rows was processed.
            # If only ads exist (no provincia_rows block ran), create the folder here.
            if provincia_folder_id is None:
                if SAVE_MODE in ("drive", "both") and drive_service and drive_root_folder_id:
                    provincia_folder_id = get_or_create_drive_folder(
                        drive_service, "provincia", drive_root_folder_id
                    )

            v_ads = get_next_provincia_version(
                "provincia_ads", flush_out_dir,
                drive_service=drive_service, drive_folder_id=provincia_folder_id
            )
            save(flush_out_dir / f"provincia_ads_{v_ads}.csv", ads_output.getvalue(),
                drive_service=drive_service, drive_folder_id=provincia_folder_id)
            logger.info(f"  -> Provincia ads CSV saved: {len(provincia_ads)} entries (version {v_ads}).")


## Run it
Entry point: Drive setup, page filter, run the pipeline, upload the log.


In [ ]:
if __name__ == "__main__":

    # ── Step 1: Drive setup first (needed for log numbering) ──
    try:
        drive_svc      = get_drive_service()
        root_folder_id = get_or_create_drive_folder(drive_svc, DRIVE_ROOT_FOLDER)
        drive_folders  = setup_drive_folders(drive_svc, root_folder_id)
        images_folder_id = drive_folders["images"]
        ocr_folder_id    = drive_folders["ocr"]
        print(f"Drive folder ready: {DRIVE_ROOT_FOLDER}")
    except Exception as e:
        print(f"Drive setup failed ({e}), running without Drive sync.")
        drive_svc        = None
        root_folder_id   = None
        images_folder_id = None
        ocr_folder_id    = None

    # ── Step 2: Get logs folder ID for Drive-aware run numbering ──
    logs_folder_id_for_logging = None
    try:
        if drive_svc and root_folder_id:
            logs_folder_id_for_logging = get_or_create_drive_folder(
                drive_svc, "logs", root_folder_id
            )
    except Exception:
        pass

    # ── Step 3: Setup logging ──
    log_path = setup_logging(
        log_dir        = Path(LOGS_DIR),
        run_mode       = RUN_MODE,
        selected_pages = SELECTED_PAGES if RUN_MODE == "pages" else None,
        page_range     = PAGE_RANGE if RUN_MODE == "range" else None,
        drive_service  = drive_svc,
        logs_folder_id = logs_folder_id_for_logging
    )
    logger = logging.getLogger("ocr_pipeline")
    logger.info(f"Log file: {log_path}")

    # ── Step 4: Build page filter ──
    if RUN_MODE == "pages":
        page_filter = set(SELECTED_PAGES)
        logger.info(f"Running {len(page_filter)} selected pages: {sorted(page_filter)}")
    elif RUN_MODE == "range":
        lo, hi = PAGE_RANGE
        page_filter = set(range(lo, hi + 1))
        logger.info(f"Running pages {lo}–{hi}: {len(page_filter)} pages")
    else:
        page_filter = None
        logger.info("Running full document.")

    # ── Step 5: Extract PDF pages ──
    all_pages = extract_pdf_pages(
        Path(PDF_PATH), dpi=200,
        drive_service   = drive_svc,
        drive_folder_id = images_folder_id,
        page_filter     = page_filter,
    )

    pages_to_run = [(name, fid) for name, fid in all_pages if fid is not None]
    logger.info(f"pages_to_run: {[name for name, _ in pages_to_run]}")
    logger.info(f"Total pages to process: {len(pages_to_run)}")

    # ── Step 6: Run OCR pipeline ──
    effective_progress = None if FORCE_REPROCESS else Path(PROGRESS_FILE)

    run_ocr(
        pages_to_run,
        progress_file        = effective_progress,
        drive_service        = drive_svc,
        drive_root_folder_id = ocr_folder_id,
    )

    # ── Step 7: Upload log to Drive logs/ folder ──
    if drive_svc and root_folder_id and log_path and log_path.exists():
        try:
            logs_folder_id = get_or_create_drive_folder(
                drive_svc, "logs", root_folder_id
            )
            upload_to_drive(drive_svc, log_path, logs_folder_id, overwrite=True)
            logger.info(f"Log uploaded to Drive logs/ folder: {log_path.name}")
        except Exception as e:
            logger.info(f"WARNING: Could not upload log to Drive: {e}")
